In [1]:
import numpy as np
import math

In [2]:
# ── Quaternion primitives ──────────────────────────────────────────────────
# Convention throughout: q = [w, x, y, z]  (scalar-first, matches MTEX q.a/b/c/d)

def quat_mul(q1: np.ndarray, q2: np.ndarray) -> np.ndarray:
    """Hamilton product of two arrays of unit quaternions.

    Supports broadcasting:  q1 (..., 4), q2 (..., 4)  →  (..., 4)
    """
    w1, x1, y1, z1 = q1[..., 0], q1[..., 1], q1[..., 2], q1[..., 3]
    w2, x2, y2, z2 = q2[..., 0], q2[..., 1], q2[..., 2], q2[..., 3]
    return np.stack([
        w1*w2 - x1*x2 - y1*y2 - z1*z2,
        w1*x2 + x1*w2 + y1*z2 - z1*y2,
        w1*y2 - x1*z2 + y1*w2 + z1*x2,
        w1*z2 + x1*y2 - y1*x2 + z1*w2,
    ], axis=-1)


def quat_inv(q: np.ndarray) -> np.ndarray:
    """Inverse of unit quaternion(s): conjugate the vector part."""
    qi = q.copy()
    qi[..., 1:] *= -1.0
    return qi


def quat_dot(q1: np.ndarray, q2: np.ndarray) -> np.ndarray:
    """Element-wise R^4 inner product (NOT Hamilton product).

    For unit quaternions:  |dot(q1, q2)| = cos(Δθ/2)
    where Δθ is the rotation-angle between q1 and q2.
    Corresponds to MTEX quat_dot(q, q_ref).
    """
    return np.einsum("...i,...i->...", q1, q2)


def quat_dot_outer(q_sub: np.ndarray, qCS: np.ndarray) -> np.ndarray:
    """Outer R^4 dot product.

    q_sub : (N, 4)
    qCS   : (M, 4)
    returns (N, M)  where result[i,j] = dot(q_sub[i], qCS[j])

    Corresponds to MTEX quat_dot_outer(q_sub, qCS).
    """
    return q_sub @ qCS.T

In [3]:
def project2FR_ref(
    q: np.ndarray,
    qCS: np.ndarray,
    q_ref: np.ndarray | None = None,
) -> np.ndarray:
    """Project quaternions into the fundamental region around a reference orientation.

    Direct Python translation of MTEX ``project2FR_ref.m``
    (geometry/@quaternion/private/project2FR_ref.m).

    Parameters
    ----------
    q : ndarray, shape (..., 4)
        Input unit quaternions [w, x, y, z] to be projected.
    qCS : ndarray, shape (M, 4)
        Proper crystal-symmetry operators [w, x, y, z].
        MTEX filters ``~qCS.i`` (improper rotations) before calling this
        function, so pass only the 24 proper rotations for cubic.
    q_ref : ndarray, shape (4,) or (N, 4) or None
        Reference orientation [w, x, y, z].
        * None  →  FR around identity (standard FZ reduction, uses |w| as
                    distance proxy, matching MTEX's ``else`` branch).
        * single (4,)  →  all grains projected toward the same reference.
        * per-grain (N, 4)  →  each grain has its own reference.

    Returns
    -------
    ndarray, same shape as ``q``
        Quaternions in the fundamental region nearest to ``q_ref``, with sign
        chosen so that ``dot(q_out, q_ref) > 0`` (or ``w > 0`` when q_ref
        is None).

    Algorithm
    ---------
    1. Compute cos(misorientation/2) to reference: ``co2 = |dot(q, q_ref)|``.
    2. Derive a threshold from the smallest non-trivial symmetry-element angle.
       Quaternions already satisfying the threshold are kept as-is (early exit
       for efficiency, identical to the MTEX ``notInside`` short-circuit).
    3. For the remaining quaternions build the (N_sub × M) matrix of distances
       ``|dot(inv(q_ref)*q_sub, qCS)|`` and pick the argmax symmetry operator.
    4. Apply the best operator: ``q_out = q_sub * inv(qCS_best)``.
    5. Flip signs so ``dot(q_out, q_ref) >= 0``.

    Notes
    -----
    * Identity: ``Re(A·B) = Re(B·A)`` for quaternions, so
      ``|dot(inv(q_ref)*q_sub, qCS)| = |dot(q_sub*inv(qCS), q_ref)|``,
      confirming the argmax finds the symmetry-equivalent nearest q_ref.
    * Quaternion convention: scalar first, [w, x, y, z], matching MTEX
      q.a / q.b / q.c / q.d.
    """
    original_shape = q.shape
    q = np.asarray(q, dtype=np.float64).reshape(-1, 4).copy()
    qCS = np.asarray(qCS, dtype=np.float64)
    N = q.shape[0]

    # ── Distance to reference ──────────────────────────────────────────────
    if q_ref is not None:
        q_ref = np.asarray(q_ref, dtype=np.float64)
        per_grain_ref = q_ref.ndim == 2  # True when q_ref is (N, 4)
        if per_grain_ref:
            q_ref_flat = q_ref.reshape(N, 4)
            co2 = np.abs(np.einsum("ni,ni->n", q, q_ref_flat))
        else:
            q_ref_flat = q_ref.reshape(4)
            co2 = np.abs(q @ q_ref_flat)
    else:
        per_grain_ref = False
        co2 = np.abs(q[:, 0])   # |w| = cos(θ/2) distance to identity

    # ── Minimum symmetry angle (threshold for "already in FR") ────────────
    # MTEX: minAngle = min non-trivial angle of qCS operators
    sym_angles = 2.0 * np.arccos(np.clip(np.abs(qCS[:, 0]), 0.0, 1.0))
    non_trivial = sym_angles[sym_angles > 1e-3]
    min_angle = float(non_trivial.min()) if non_trivial.size > 0 else np.inf

    not_inside = co2 < np.cos(min_angle / 4.0)

    # ── Project quaternions not yet in FR ─────────────────────────────────
    if np.any(not_inside) and qCS.shape[0] > 1:
        q_sub = q[not_inside]           # (K, 4)

        if q_ref is not None:
            if per_grain_ref:
                # Each grain in q_sub has its own reference
                q_ref_sub = q_ref_flat[not_inside]          # (K, 4)
                iq_ref_sub = quat_inv(q_ref_sub)            # (K, 4)
                lhs = quat_mul(iq_ref_sub, q_sub)           # (K, 4)
            else:
                # Single reference for all grains
                iq_ref = quat_inv(q_ref_flat[np.newaxis])   # (1, 4)
                lhs = quat_mul(np.broadcast_to(iq_ref, q_sub.shape).copy(), q_sub)  # (K, 4)
            # omegaSym[i, j] = |dot(inv(q_ref[i]) * q_sub[i], qCS[j])|
            omegaSym = np.abs(quat_dot_outer(lhs, qCS))     # (K, M)
        else:
            omegaSym = np.abs(quat_dot_outer(q_sub, qCS))   # (K, M)

        # Best symmetry operator index for each quaternion in q_sub
        nx = np.argmax(omegaSym, axis=1)                    # (K,)

        # Project:  q_out = q_sub * inv(qCS[nx])
        qCS_best_inv = quat_inv(qCS[nx])                    # (K, 4)
        q[not_inside] = quat_mul(q_sub, qCS_best_inv)

    # ── Ensure sign consistency with reference ────────────────────────────
    if q_ref is None:
        change_sign = q[:, 0] < 0
    elif per_grain_ref:
        change_sign = np.einsum("ni,ni->n", q, q_ref_flat) < 0
    else:
        change_sign = (q @ q_ref_flat) < 0

    q[change_sign] *= -1.0

    return q.reshape(original_shape)

In [4]:
# ── FCC / cubic-O proper symmetry operators (24 rotations) ───────────────
# Source: fcc_syms_inv.py  →  build_fcc_syms()
# These are the same operators MTEX uses after filtering ~qCS.i (improper rotations).

def build_fcc_syms() -> np.ndarray:
    s = 1.0 / math.sqrt(2.0)
    h = 0.5
    return np.array([
        [1,  0,  0,  0],
        [0,  1,  0,  0],
        [0,  0,  1,  0],
        [0,  0,  0,  1],
        [s,  s,  0,  0],
        [s,  0,  s,  0],
        [s,  0,  0,  s],
        [s, -s,  0,  0],
        [s,  0, -s,  0],
        [s,  0,  0, -s],
        [0,  s,  s,  0],
        [0,  s,  0,  s],
        [0,  0,  s,  s],
        [0,  s, -s,  0],
        [0,  0,  s, -s],
        [0,  s,  0, -s],
        [h,  h,  h,  h],
        [h, -h, -h,  h],
        [h, -h,  h, -h],
        [h,  h, -h, -h],
        [h,  h,  h, -h],
        [h,  h, -h,  h],
        [h, -h,  h,  h],
        [h, -h, -h, -h],
    ], dtype=np.float64)

qCS = build_fcc_syms()
print(f"qCS shape: {qCS.shape}  (should be (24, 4))")

qCS shape: (24, 4)  (should be (24, 4))


In [5]:
# ── Smoke tests ──────────────────────────────────────────────────────────

rng = np.random.default_rng(42)

def random_unit_quats(n: int, rng) -> np.ndarray:
    q = rng.standard_normal((n, 4))
    return q / np.linalg.norm(q, axis=-1, keepdims=True)

# --- Test 1: q_ref=None (standard FZ reduction) --------------------------
# Identity should map to itself; all outputs should have w >= 0.
q_test = random_unit_quats(1000, rng)
q_fz = project2FR_ref(q_test, qCS, q_ref=None)

assert q_fz.shape == q_test.shape
assert np.all(q_fz[:, 0] >= 0), "w should be non-negative after FZ reduction"

# Each output should be a valid symmetry-equivalent of the input.
# Check: for each i, min_j |dot(q_fz[i], qCS[j] * q_test[i])| should be close to 1
# (i.e. q_fz[i] == qCS[j] * q_test[i] for some j, up to sign)
equiv = np.abs(quat_dot_outer(q_fz, qCS))  # (N, 24) — dot with sym * q_fz would need Hamilton; use a simpler check:
# Actually check that q_fz and q_test agree on their FZ representative by re-projecting
q_fz2 = project2FR_ref(q_fz, qCS, q_ref=None)
diff = np.abs(np.abs(np.einsum("ni,ni->n", q_fz, q_fz2)) - 1.0)
assert diff.max() < 1e-10, f"Idempotency failed: max_diff={diff.max():.2e}"
print(f"Test 1 PASSED — FZ reduction idempotent, w≥0 enforced, N={len(q_test)}")

# --- Test 2: single q_ref ------------------------------------------------
q_ref_single = random_unit_quats(1, rng)[0]
q_fz_ref = project2FR_ref(q_test, qCS, q_ref=q_ref_single)

# dot(q_out, q_ref) should be >= 0 for all outputs
dots = q_fz_ref @ q_ref_single
assert np.all(dots >= 0), "dot(q_out, q_ref) should be non-negative"
# Idempotency: re-projecting should change nothing
q_fz_ref2 = project2FR_ref(q_fz_ref, qCS, q_ref=q_ref_single)
diff2 = np.abs(np.abs(np.einsum("ni,ni->n", q_fz_ref, q_fz_ref2)) - 1.0)
assert diff2.max() < 1e-10, f"Idempotency (ref) failed: max_diff={diff2.max():.2e}"
print(f"Test 2 PASSED — Single q_ref: dot≥0 enforced, idempotent, N={len(q_test)}")

# --- Test 3: per-grain q_ref ---------------------------------------------
q_refs_per_grain = random_unit_quats(len(q_test), rng)
q_fz_pg = project2FR_ref(q_test, qCS, q_ref=q_refs_per_grain)
dots_pg = np.einsum("ni,ni->n", q_fz_pg, q_refs_per_grain)
assert np.all(dots_pg >= 0), "Per-grain: dot(q_out, q_ref[i]) should be non-negative"
print(f"Test 3 PASSED — Per-grain q_ref: dot≥0 enforced, N={len(q_test)}")

# --- Test 4: known symmetry equivalent -----------------------------------
# Take a quaternion in the FZ, apply a non-trivial symmetry, re-reduce → should get back original.
q0 = project2FR_ref(random_unit_quats(50, rng), qCS)                   # put 50 in FZ
sym_op = qCS[5]                                                          # some non-identity sym
q_equiv = quat_mul(q0, sym_op[np.newaxis])                              # apply sym
q_back = project2FR_ref(q_equiv, qCS)                                    # reduce back
same = np.abs(np.abs(np.einsum("ni,ni->n", q0, q_back)) - 1.0) < 1e-10
print(f"Test 4 — sym-equiv recovery: {same.sum()}/{len(q0)} matched (expected ~all)")

print("\nAll tests passed.")

Test 1 PASSED — FZ reduction idempotent, w≥0 enforced, N=1000
Test 2 PASSED — Single q_ref: dot≥0 enforced, idempotent, N=1000
Test 3 PASSED — Per-grain q_ref: dot≥0 enforced, N=1000
Test 4 — sym-equiv recovery: 50/50 matched (expected ~all)

All tests passed.


## Passive Convention (Bunge)

In [6]:
# ── Bunge (passive) ↔ active conversion ──────────────────────────────────
#
# MTEX / the code above uses the ACTIVE convention:
#   q_active  → rotates crystal vectors expressed in the sample frame.
#
# Bunge Euler angles define a PASSIVE rotation (a.k.a. "crystal-to-sample"
# frame change).  The corresponding unit quaternion is the CONJUGATE:
#
#       q_bunge = conj(q_active)  =  [w, -x, -y, -z]
#       q_active = conj(q_bunge)
#
# Consequence for FZ reduction:
#   Active FZ:  q_fz = q * qCS⁻¹   (symmetry applied on the RIGHT with inverse)
#   Bunge  FZ:  q_fz = qCS * q     (symmetry applied on the LEFT directly)
#
# Therefore the cleanest approach is:
#   1. Convert Bunge → active  (conjugate vector part)
#   2. Call project2FR_ref     (unchanged)
#   3. Convert active → Bunge  (conjugate vector part again)

def bunge_to_active(q_b: np.ndarray) -> np.ndarray:
    """Bunge passive quaternion → active quaternion (conjugate vector part)."""
    q_a = q_b.copy()
    q_a[..., 1:] *= -1.0
    return q_a

active_to_bunge = bunge_to_active   # same operation; alias for clarity


def project2FR_ref_bunge(
    q_b: np.ndarray,
    qCS: np.ndarray,
    q_ref_b: np.ndarray | None = None,
) -> np.ndarray:
    """FZ reduction for Bunge (passive) quaternions.

    Parameters
    ----------
    q_b     : (..., 4)  Input Bunge quaternions [w, x, y, z].
    qCS     : (M, 4)    Proper crystal-symmetry operators (same as active version).
    q_ref_b : (4,) or (N, 4) or None
              Reference orientation in Bunge convention.
              None → standard FZ around identity (|w| maximised).

    Returns
    -------
    ndarray, same shape as q_b, Bunge quaternions in the fundamental region.
    """
    # Convert inputs to active
    q_a = bunge_to_active(q_b)
    q_ref_a = bunge_to_active(q_ref_b) if q_ref_b is not None else None

    # FZ reduction in active convention (existing function, unchanged)
    q_a_fz = project2FR_ref(q_a, qCS, q_ref=q_ref_a)

    # Convert result back to Bunge
    return active_to_bunge(q_a_fz)


In [7]:
# ── Smoke tests (Bunge convention) ───────────────────────────────────────

rng_b = np.random.default_rng(99)

def random_unit_quats_b(n, rng):
    q = rng.standard_normal((n, 4))
    return q / np.linalg.norm(q, axis=-1, keepdims=True)

q_bunge = random_unit_quats_b(1000, rng_b)

# --- Test B1: q_ref=None, idempotency ------------------------------------
q_b_fz = project2FR_ref_bunge(q_bunge, qCS, q_ref_b=None)
assert q_b_fz.shape == q_bunge.shape
assert np.all(q_b_fz[:, 0] >= 0), "w should be non-negative (FZ around identity)"

q_b_fz2 = project2FR_ref_bunge(q_b_fz, qCS, q_ref_b=None)
diff = np.abs(np.abs(np.einsum("ni,ni->n", q_b_fz, q_b_fz2)) - 1.0)
assert diff.max() < 1e-10, f"Idempotency failed: {diff.max():.2e}"
print(f"Test B1 PASSED — FZ reduction idempotent in Bunge convention, N={len(q_bunge)}")

# --- Test B2: single q_ref -----------------------------------------------
q_ref_b_single = random_unit_quats_b(1, rng_b)[0]
q_b_fz_ref = project2FR_ref_bunge(q_bunge, qCS, q_ref_b=q_ref_b_single)

dots = q_b_fz_ref @ q_ref_b_single
assert np.all(dots >= 0), "dot(q_b_out, q_ref_b) should be non-negative"
q_b_fz_ref2 = project2FR_ref_bunge(q_b_fz_ref, qCS, q_ref_b=q_ref_b_single)
diff2 = np.abs(np.abs(np.einsum("ni,ni->n", q_b_fz_ref, q_b_fz_ref2)) - 1.0)
assert diff2.max() < 1e-10
print(f"Test B2 PASSED — Single Bunge q_ref: dot≥0 enforced, idempotent")

# --- Test B3: result matches round-trip through active -------------------
# project2FR_ref_bunge must give the same answer as:
#   convert to active → reduce → convert back
q_a_rt = bunge_to_active(q_bunge)
q_a_fz_rt = project2FR_ref(q_a_rt, qCS)
q_b_fz_rt = active_to_bunge(q_a_fz_rt)
same = np.abs(np.abs(np.einsum("ni,ni->n", q_b_fz, q_b_fz_rt)) - 1.0) < 1e-10
assert same.all(), f"Round-trip mismatch: {(~same).sum()} grains differ"
print(f"Test B3 PASSED — Bunge wrapper matches manual round-trip, N={len(q_bunge)}")

print("\nAll Bunge tests passed.")


Test B1 PASSED — FZ reduction idempotent in Bunge convention, N=1000
Test B2 PASSED — Single Bunge q_ref: dot≥0 enforced, idempotent
Test B3 PASSED — Bunge wrapper matches manual round-trip, N=1000

All Bunge tests passed.


## Make project2FR_ref_bunge function from scratch.
### Dfiffernce is left multiply with sym^-1--> sym^-1 x q

In [8]:
# ── From-scratch FZ reduction in Bunge (passive) convention ──────────────
#
# Bunge symmetry equivalents of q_b:  { qCS[j]^{-1} * q_b  :  j = 1..M }
#                                         ↑ LEFT multiply by inverse
#
# Distance metrics (derived via left-multiply isometry: dot(a*p, a*q) = dot(p,q)):
#
#   q_ref=None :  |w( qCS[j]^{-1} * q_b )|  =  |dot(qCS[j], q_b)|
#                 (same formula as active — only the apply step differs)
#
#   q_ref given:  |dot( qCS[j]^{-1} * q_b, q_ref_b )|
#                 = |dot( q_b, qCS[j] * q_ref_b )|    ← isometry with a=qCS[j]
#                 pre-compute  qCS_qref[j] = qCS[j] * q_ref_b  →  (M,4)
#                 then  omegaSym[i,j] = |dot(q_b[i], qCS_qref[j])|

def project2FR_ref_bunge_scratch(
    q_b: np.ndarray,
    qCS: np.ndarray,
    q_ref_b: np.ndarray | None = None,
) -> np.ndarray:
    """FZ reduction for Bunge (passive) quaternions — built from scratch.

    Parameters
    ----------
    q_b     : (..., 4)  Input Bunge quaternions [w, x, y, z].
    qCS     : (M, 4)    Proper crystal-symmetry operators.
    q_ref_b : (4,) or (N, 4) or None
              Reference orientation in Bunge convention.
              None → standard FZ around identity (|w| maximised).

    Returns
    -------
    ndarray, same shape as q_b, Bunge quaternions in the fundamental region.

    Key difference vs. active version
    ----------------------------------
      Active apply:  q_fz = q_sub * qCS_best^{-1}   (right multiply)
      Bunge  apply:  q_fz = qCS_best^{-1} * q_sub   (LEFT multiply)
    """
    original_shape = q_b.shape
    q = np.asarray(q_b, dtype=np.float64).reshape(-1, 4).copy()
    qCS = np.asarray(qCS, dtype=np.float64)
    N = q.shape[0]

    # ── Distance to reference ──────────────────────────────────────────────
    if q_ref_b is not None:
        q_ref = np.asarray(q_ref_b, dtype=np.float64)
        per_grain_ref = q_ref.ndim == 2
        q_ref_flat = q_ref.reshape(N, 4) if per_grain_ref else q_ref.reshape(4)
        co2 = (np.abs(np.einsum("ni,ni->n", q, q_ref_flat))
               if per_grain_ref else np.abs(q @ q_ref_flat))
    else:
        per_grain_ref = False
        co2 = np.abs(q[:, 0])   # |w(qCS_id^{-1} * q)| = |q[0]|

    # ── Threshold ─────────────────────────────────────────────────────────
    sym_angles  = 2.0 * np.arccos(np.clip(np.abs(qCS[:, 0]), 0.0, 1.0))
    non_trivial = sym_angles[sym_angles > 1e-3]
    min_angle   = float(non_trivial.min()) if non_trivial.size > 0 else np.inf
    not_inside  = co2 < np.cos(min_angle / 4.0)

    # ── Project quaternions not yet in FR ─────────────────────────────────
    if np.any(not_inside) and qCS.shape[0] > 1:
        q_sub = q[not_inside]   # (K, 4)

        if q_ref_b is not None:
            if per_grain_ref:
                q_ref_sub = q_ref_flat[not_inside]               # (K, 4)
                # qCS_qref[i,j] = qCS[j] * q_ref_b[i]  →  (K, M, 4)
                qCS_qref  = quat_mul(qCS[np.newaxis],            # (1, M, 4)
                                     q_ref_sub[:, np.newaxis])   # (K, 1, 4)
                omegaSym  = np.abs(np.einsum("ki,kmi->km", q_sub, qCS_qref))  # (K, M)
            else:
                # qCS_qref[j] = qCS[j] * q_ref_b  →  (M, 4)
                qCS_qref = quat_mul(qCS, q_ref_flat[np.newaxis])  # (M, 4)
                omegaSym = np.abs(q_sub @ qCS_qref.T)             # (K, M)
        else:
            # |w(qCS[j]^{-1} * q_sub)| = |dot(qCS[j], q_sub)|
            omegaSym = np.abs(quat_dot_outer(q_sub, qCS))         # (K, M)

        nx = np.argmax(omegaSym, axis=1)                           # (K,)

        # ── Bunge apply: LEFT multiply by qCS_best^{-1} ───────────────────
        qCS_best_inv      = quat_inv(qCS[nx])                      # (K, 4)
        q[not_inside]     = quat_mul(qCS_best_inv, q_sub)          # ← LEFT

    # ── Sign consistency with reference ───────────────────────────────────
    if q_ref_b is None:
        change_sign = q[:, 0] < 0
    elif per_grain_ref:
        change_sign = np.einsum("ni,ni->n", q, q_ref_flat) < 0
    else:
        change_sign = (q @ q_ref_flat) < 0

    q[change_sign] *= -1.0
    return q.reshape(original_shape)


In [9]:
# ── Tests: from-scratch matches the wrapper, all three q_ref modes ────────

rng_s = np.random.default_rng(7)
q_b_test = random_unit_quats_b(1000, rng_s)

# --- Test S1: q_ref=None — matches wrapper, idempotent -------------------
q_wrap   = project2FR_ref_bunge(q_b_test, qCS, q_ref_b=None)
q_scratch = project2FR_ref_bunge_scratch(q_b_test, qCS, q_ref_b=None)

match = np.abs(np.abs(np.einsum("ni,ni->n", q_wrap, q_scratch)) - 1.0) < 1e-10
assert match.all(), f"S1 mismatch: {(~match).sum()} grains"
assert np.all(q_scratch[:, 0] >= 0)

q_scratch2 = project2FR_ref_bunge_scratch(q_scratch, qCS, q_ref_b=None)
idem = np.abs(np.abs(np.einsum("ni,ni->n", q_scratch, q_scratch2)) - 1.0) < 1e-10
assert idem.all()
print(f"Test S1 PASSED — q_ref=None: matches wrapper, idempotent, N={len(q_b_test)}")

# --- Test S2: single q_ref_b — matches wrapper ---------------------------
q_ref_b_s = random_unit_quats_b(1, rng_s)[0]
q_wrap_r   = project2FR_ref_bunge(q_b_test, qCS, q_ref_b=q_ref_b_s)
q_scr_r    = project2FR_ref_bunge_scratch(q_b_test, qCS, q_ref_b=q_ref_b_s)

match2 = np.abs(np.abs(np.einsum("ni,ni->n", q_wrap_r, q_scr_r)) - 1.0) < 1e-10
assert match2.all(), f"S2 mismatch: {(~match2).sum()} grains"
assert np.all(q_scr_r @ q_ref_b_s >= 0)
print(f"Test S2 PASSED — single q_ref_b: matches wrapper, dot≥0 enforced")

# --- Test S3: per-grain q_ref_b — matches wrapper ------------------------
q_refs_pg  = random_unit_quats_b(len(q_b_test), rng_s)
q_wrap_pg  = project2FR_ref_bunge(q_b_test, qCS, q_ref_b=q_refs_pg)
q_scr_pg   = project2FR_ref_bunge_scratch(q_b_test, qCS, q_ref_b=q_refs_pg)

match3 = np.abs(np.abs(np.einsum("ni,ni->n", q_wrap_pg, q_scr_pg)) - 1.0) < 1e-10
assert match3.all(), f"S3 mismatch: {(~match3).sum()} grains"
dots_pg = np.einsum("ni,ni->n", q_scr_pg, q_refs_pg)
assert np.all(dots_pg >= 0)
print(f"Test S3 PASSED — per-grain q_ref_b: matches wrapper, dot≥0 enforced")

print("\nAll from-scratch tests passed.")


Test S1 PASSED — q_ref=None: matches wrapper, idempotent, N=1000
Test S2 PASSED — single q_ref_b: matches wrapper, dot≥0 enforced
Test S3 PASSED — per-grain q_ref_b: matches wrapper, dot≥0 enforced

All from-scratch tests passed.
